# Documentação — 09 - Ingestão Gold: Cálculo de KPIs e Persistência
**Squad 2 — Real Time for Business | Dupla 2**  
**Integrantes:** Kauan & Leandro  
**Tabelas de Escopo:** KPIs derivados de `silver_ecommerce_pedidos`  
**Branch:** `feat/squad2/kauan-leandro`  

Documentação técnica do notebook de Ingestão Gold — cálculo e persistência de KPIs de negócio a partir da camada Silver.

## Visão geral

| KPI | Tabela Gold (Delta) | Tabela SQL Server | Modo | Registros (1ª execução) |
| --- | --- | --- | --- | --- |
| Volume de Pedidos (5 min) | `gold_kpi_volume_pedidos_5min` | `gold_kpi_volume_pedidos_5min_dupla_2` | Overwrite | 65.504 |
| Ticket Médio por Pagamento | `gold_kpi_ticket_medio_por_pagamento` | `gold_kpi_ticket_medio_por_pagamento_dupla_2` | Overwrite | 4 |
| Taxa de Cancelamento (hora) | `gold_kpi_taxa_cancelamento_hora` | `gold_kpi_taxa_cancelamento_hora_dupla_2` | Overwrite | 17.528 |

## Por que estes KPIs?

A camada Gold tem como objetivo transformar os dados validados da Silver em **métricas de negócio acionáveis** para tomada de decisão em tempo real:

* **Volume de pedidos (5 min):** permite detectar picos de demanda em tempo real e dimensionar recursos operacionais (estoque, logística, atendimento)
* **Ticket médio por pagamento:** identifica quais métodos de pagamento geram maior receita média, orientando promoções e parcerias com operadores de pagamento
* **Taxa de cancelamento (hora):** detecta horas do dia com maior taxa de cancelamento, permitindo investigar causas (falhas no checkout, problemas de entrega, etc.) e agir proativamente

## Estrutura do notebook (13 células)

| Célula | Tipo | Função |
| --- | --- | --- |
| 1 | Markdown | Título, KPIs calculados, tabelas geradas e fluxo de execução |
| 2 | Run | `%run ./00_setup_config` — carrega credenciais `.env` |
| 3 | Python | Imports + config JDBC + funções auxiliares (`tabela_existe`, `escrever_jdbc`, `escrever_delta`) + nomes das tabelas |
| 4 | Markdown | KPI 1 — Volume de pedidos (5 min) |
| 5 | Python | Cálculo do KPI 1: `F.window("dt_pedido", "5 minutes")` com count, sum e avg |
| 6 | Markdown | KPI 2 — Ticket médio por pagamento |
| 7 | Python | Cálculo do KPI 2: normalização `metodo_pagamento` (upper + trim) + groupBy |
| 8 | Markdown | KPI 3 — Taxa de cancelamento por hora |
| 9 | Python | Cálculo do KPI 3: `F.window("dt_pedido", "1 hour")` + `status_pedido == 'Cancelado'` |
| 10 | Markdown | Persistência — explicação da estratégia overwrite + append |
| 11 | Python | Persistência: Delta (overwrite KPIs + append histórico) + SQL Server (overwrite, flag `EXECUTAR_JDBC`) |
| 12 | Markdown | Resumo geral |
| 13 | Python | Resumo executável: tabela consolidada + contagem das tabelas Gold e histórico |

## Arquitetura

```
Silver (Unity Catalog)          Gold (Delta - UC)              SQL Server (JDBC)
┌────────────────────┐         ┌──────────────────┐          ┌──────────────────────┐
│ silver_ecommerce_  │────────>│ gold_kpi_*        │─────────>│ gold_kpi_*_dupla_2   │
│     pedidos        │  calc   │ (overwrite)       │ overwrite│                      │
│                    │         │                   │          │                      │
│                    │         │ gold_hist_kpi_*   │          │                      │
│                    │────────>│ (append)          │          │                      │
└────────────────────┘         └──────────────────┘          └──────────────────────┘
```

**Leitura:** `workspace.default.silver_ecommerce_pedidos` (90.343 registros)
**Escrita Delta:** `workspace.default.gold_kpi_*` (overwrite) + `workspace.default.gold_hist_kpi_*` (append)
**Escrita JDBC:** SQL Server com sufixo `_dupla_2` para isolamento entre duplas da Squad 2

---
# KPI 1 — Volume de Pedidos (janela de 5 minutos)

## Objetivo
Contar o número de pedidos e somar o valor total em janelas de 5 minutos sobre `dt_pedido`, permitindo detectar picos de demanda em tempo real.

## Origem
`workspace.default.silver_ecommerce_pedidos` — 90.343 registros

## Colunas da origem utilizadas
| Coluna | Tipo | Uso |
| --- | --- | --- |
| `dt_pedido` | timestamp | Chave da janela temporal (5 min) |
| `id_pedido` | long | Contagem (COUNT) |
| `valor_total` | double | Soma (SUM) e média (AVG) |

## Colunas geradas
| Coluna | Tipo | Descrição |
| --- | --- | --- |
| `dt_inicio_janela` | timestamp | Início da janela de 5 min |
| `dt_fim_janela` | timestamp | Fim da janela de 5 min |
| `qtd_pedidos` | long | Número de pedidos na janela |
| `valor_total_pedidos` | double | Soma de `valor_total` na janela |
| `ticket_medio` | double | Valor médio por pedido na janela |
| `dt_calculo` | timestamp | Momento do cálculo (para histórico) |

## Fórmula
```python
F.window("dt_pedido", "5 minutes")  # janela temporal
F.count("id_pedido")                 # quantidade de pedidos
F.sum("valor_total")                 # valor total
F.avg("valor_total")                 # ticket médio
```

## Resultado (1ª execução)
* **65.504 janelas** de 5 minutos calculadas
* Período coberto: 2024-01-01 a 2026-09-21
* Persistido em: `workspace.default.gold_kpi_volume_pedidos_5min` (overwrite) + `gold_hist_volume_pedidos_5min` (append)
* Tabela SQL Server: `gold_kpi_volume_pedidos_5min_dupla_2` (overwrite)

---
# KPI 2 — Ticket Médio por Método de Pagamento

## Objetivo
Calcular o ticket médio (valor médio por pedido) agrupado por método de pagamento, permitindo identificar quais métodos geram maior receita.

## Origem
`workspace.default.silver_ecommerce_pedidos` — 90.343 registros

## Colunas da origem utilizadas
| Coluna | Tipo | Uso |
| --- | --- | --- |
| `metodo_pagamento` | string | Chave de agrupamento (após normalização) |
| `id_pedido` | long | Contagem (COUNT) |
| `valor_total` | double | Soma (SUM) e média (AVG) |

## Por que normalizar?
Os dados de origem contêm variações de case: `"Pix"` vs `"pix"`, `"Boleto"` vs `"boleto"`. A normalização (`upper + trim`) agrupa registros equivalentes:

| Valor original | Valor normalizado | Registros |
| --- | --- | --- |
| `"Cartão de Crédito"` | `CARTÃO DE CRÉDITO` | 58.662 |
| `"Pix"` + `"pix"` | `PIX` | 27.180 |
| `"Boleto"` + `"boleto"` | `BOLETO` | 4.499 |
| `"cartão débito"` | `CARTÃO DÉBITO` | 2 |

## Colunas geradas
| Coluna | Tipo | Descrição |
| --- | --- | --- |
| `metodo_pagamento` | string | Método normalizado (upper + trim) |
| `qtd_pedidos` | long | Número de pedidos por método |
| `valor_total` | double | Soma de `valor_total` por método |
| `ticket_medio` | double | Valor médio por pedido |
| `dt_calculo` | timestamp | Momento do cálculo |

## Resultado (1ª execução)
| Método | Qtd Pedidos | Valor Total | Ticket Médio |
| --- | --- | --- | --- |
| CARTÃO DE CRÉDITO | 58.662 | R$ 49.410.613,01 | R$ 842,29 |
| PIX | 27.180 | R$ 22.924.473,42 | R$ 843,43 |
| BOLETO | 4.499 | R$ 3.733.557,25 | R$ 829,86 |
| CARTÃO DÉBITO | 2 | R$ 340,31 | R$ 170,16 |
* Persistido em: `workspace.default.gold_kpi_ticket_medio_por_pagamento` (overwrite) + `gold_hist_ticket_medio_por_pagamento` (append)
* Tabela SQL Server: `gold_kpi_ticket_medio_por_pagamento_dupla_2` (overwrite)

---
# KPI 3 — Taxa de Cancelamento por Hora

## Objetivo
Calcular a porcentagem de pedidos cancelados por hora, permitindo detectar períodos críticos do dia.

## Origem
`workspace.default.silver_ecommerce_pedidos` — 90.343 registros

## Colunas da origem utilizadas
| Coluna | Tipo | Uso |
| --- | --- | --- |
| `dt_pedido` | timestamp | Chave da janela temporal (1 hora) |
| `id_pedido` | long | Contagem total |
| `status_pedido` | string | Identifica cancelados (`= 'Cancelado'`) |

## Colunas geradas
| Coluna | Tipo | Descrição |
| --- | --- | --- |
| `dt_inicio_janela` | timestamp | Início da janela de 1 hora |
| `dt_fim_janela` | timestamp | Fim da janela de 1 hora |
| `total_pedidos` | long | Total de pedidos na hora |
| `pedidos_cancelados` | long | Pedidos com `status_pedido = 'Cancelado'` |
| `taxa_cancelamento` | double | % de cancelamento (cancelados / total * 100) |
| `dt_calculo` | timestamp | Momento do cálculo |

## Fórmula
```python
F.window("dt_pedido", "1 hour")
F.count("id_pedido")                                          # total
F.sum(F.when(F.col("status_pedido") == "Cancelado", 1)
     .otherwise(0))                                          # cancelados
F.round(F.col("pedidos_cancelados") / F.col("total_pedidos") * 100, 2)  # taxa %
```

## Status válidos na origem
`Entregue` (54.175), `Enviado` (13.608), `Em Separação` (9.007), `Cancelado` (4.555), `Pagamento Aprovado` (4.553), `Processando` (4.444), `Aguardando` (1)

## Resultado (1ª execução)
* **17.528 janelas** de 1 hora calculadas
* Exemplo: janela 2024-01-01 08:00 → 12 pedidos, 2 cancelados = **16,67%**
* Persistido em: `workspace.default.gold_kpi_taxa_cancelamento_hora` (overwrite) + `gold_hist_taxa_cancelamento_hora` (append)
* Tabela SQL Server: `gold_kpi_taxa_cancelamento_hora_dupla_2` (overwrite)

---
# Persistência — Delta + SQL Server

## Estratégia de escrita

| Destino | Tabela | Modo | Por que? |
| --- | --- | --- | --- |
| Delta (UC) | `gold_kpi_*` | **Overwrite** | KPI é estado atual — a cada recálculo, substitui o snapshot anterior |
| Delta (UC) | `gold_hist_kpi_*` | **Append** | Histórico acumula versões ao longo do tempo para análise de evolução |
| SQL Server | `gold_kpi_*_dupla_2` | **Overwrite** | Espelha o KPI atual no banco relacional, acessível por BI e outras ferramentas |

## Por que `overwrite` nos KPIs?
KPIs representam o **estado atual** do negócio. A cada ciclo de polling, os valores são recalculados com a janela de tempo mais recente. O `overwrite` substitui os dados antigos pelo snapshot atual, mantendo apenas a versão mais recente.

## Por que `append` no histórico?
A tabela de histórico acumula cada execução, permitindo:
* Comparar a evolução dos KPIs entre ciclos
* Detectar tendências (ex: ticket médio crescendo ao longo do dia)
* Auditoria de quando cada KPI foi calculado (via `dt_calculo`)

## Por que escrever no SQL Server?
O SQL Server é o banco de dados compartilhado do estágio, onde os dados originais residem. Escrever os KPIs de volta permite:
* Consulta via SQL diretamente no banco
* Integração com ferramentas de BI externas
* Comparação entre os dados originais e os KPIs calculados

## Por que o sufixo `_dupla_2`?
Outras duplas da Squad 2 receberam a mesma tarefa com os mesmos nomes de tabelas. Sem o sufixo, uma dupla sobrescreveria os dados da outra no SQL Server (schema `dbo` compartilhado). O sufixo `_dupla_2` garante isolamento.

## Flag `EXECUTAR_JDBC`
A célula de persistência contém a flag `EXECUTAR_JDBC = False`. Para habilitar a escrita no SQL Server:
1. Abra o notebook `09_ingestao_gold_kpis`
2. Altere para `EXECUTAR_JDBC = True` na célula 11
3. Execute a célula

**Por que `False` por padrão?** A escrita JDBC é uma operação sensível (banco externo compartilhado). O `False` evita execução acidental e permite testar apenas as escritas Delta primeiro.

## Funções auxiliares

| Função | Descrição | Por que existe? |
| --- | --- | --- |
| `tabela_existe(nome)` | Verifica se uma tabela existe no catálogo | Usada no resumo para checar se as tabelas Gold foram criadas |
| `escrever_delta(df, tabela, modo)` | Escreve DataFrame em Delta no UC | Centraliza a lógica de escrita com verificação de DataFrame vazio |
| `escrever_jdbc(df, tabela, modo)` | Escreve DataFrame no SQL Server via JDBC | Centraliza a lógica JDBC com verificação de segurança (count > 0 antes de escrever) |

---
# Resumo Geral — Status dos KPIs Gold

## Tabelas criadas no Unity Catalog

| Tabela | Tipo | Registros | Modo |
| --- | --- | --- | --- |
| `workspace.default.gold_kpi_volume_pedidos_5min` | KPI | 65.504 | overwrite |
| `workspace.default.gold_kpi_ticket_medio_por_pagamento` | KPI | 4 | overwrite |
| `workspace.default.gold_kpi_taxa_cancelamento_hora` | KPI | 17.528 | overwrite |
| `workspace.default.gold_hist_volume_pedidos_5min` | Histórico | 65.504 | append |
| `workspace.default.gold_hist_ticket_medio_por_pagamento` | Histórico | 4 | append |
| `workspace.default.gold_hist_taxa_cancelamento_hora` | Histórico | 17.528 | append |

## Tabelas no SQL Server (pendentes — habilitar `EXECUTAR_JDBC = True`)

| Tabela | Modo |
| --- | --- |
| `gold_kpi_volume_pedidos_5min_dupla_2` | overwrite |
| `gold_kpi_ticket_medio_por_pagamento_dupla_2` | overwrite |
| `gold_kpi_taxa_cancelamento_hora_dupla_2` | overwrite |

## Pipeline completo do estágio (Bronze → Silver → Gold)

```
Bronze (04)                Silver (08)                 Gold (09)
┌─────────────────┐       ┌─────────────────┐        ┌─────────────────┐
│ controle_pedidos │──────>│ silver_pedidos   │───────>│ gold_kpi_*       │
│ pedidos_filtrados│       │ silver_itens    │        │ gold_hist_kpi_*  │
└─────────────────┘       │ silver_produtos  │        └────────┬────────┘
                           │ silver_categorias│                 │
                           │ silver_rastream. │                 v
                           │ silver_enderecos │          SQL Server (JDBC)
                           │ silver_clientes   │          gold_kpi_*_dupla_2
                           └─────────────────┘
```

| Camada | Notebook | Função |
| --- | --- | --- |
| Bronze | `04_ingestao_bronze` | Controle de hash SHA-256 (missão 01) + clientes ADLS |
| Silver | `08_ingestao_silver_all` | Validação de qualidade (missões 02, 03, 04, 05, 07) |
| Silver | `05_ingestao_silver` | Validação de clientes (missão 06, Kauan) |
| **Gold** | **`09_ingestao_gold_kpis`** | **Cálculo de KPIs + persistência Delta + SQL Server** |